<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Waterloo-Park-LiDAR-Tree-Detection-Pipeline
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A Machine Learning Exploration of LiDAR Classification
</h2>
</div>

##### Version Number: 1.0
---
### Contents  
---
### Notes
---
### Inputs
---
### Outputs  
---
### User Created Dependencies  
---
### Third Party Dependencies

In [142]:
import pandas as pd
import numpy as np
import laspy
import rasterio
from rasterio.transform import rowcol

### Load LAS file

In [ ]:
# Load the LAS/LAZ file
las = laspy.read("data/clipped_point_cloud.las")

cloud = pd.DataFrame({
    "X": np.array(las.x),
    "Y": np.array(las.y),
    "Z": np.array(las.z),
    "intensity": np.array(las.intensity),
    "classification": np.array(las.classification)
})

point_ids = np.arange(len(cloud))

### Load Orthophoto Raster

In [172]:
with rasterio.open("data/clipped_ortho.tif") as src:
    transform = src.transform
    bounds = src.bounds
    crs = src.crs
    
    x_min = bounds.left
    y_min = bounds.bottom
    x_max = bounds.right
    y_max = bounds.top

    width = src.width
    height = src.height

    red_band = src.read(1)  # Red
    green_band = src.read(2)  # Green
    blue_band = src.read(3)  # Blue

Isolate Ground Points

In [192]:
ground = cloud[cloud.classification == 2]

In [193]:
gx = ground.X
gy = ground.Y
gz = ground.Z

Group ground points in grid of orthophoto

In [ ]:
ground = cloud[cloud.classification == 2]

rows_idx, cols_idx = rasterio.transform.rowcol(
    transform,
    ground.X,
    ground.Y
)

## eliminate points outside bounds
mask = (
    (rows_idx >= 0) & (rows_idx < height) &
    (cols_idx >= 0) & (cols_idx < width)
)

ground_in_ortho = ground[mask]

In [ ]:
gr = pd.DataFrame({
    "row": rows_idx,
    "col": cols_idx,
    "z": ground.Z.values
})

In [ ]:
## Find mean elevation in each individual grid
mean_ground_grid = gr.groupby(["row", "col"])["z"].mean().reset_index()

Filter only unclassified points

In [ ]:
## use unclassified points only
trees = cloud[cloud.classification == 1]

Group unclassified points by position in orthophoto raster

In [ ]:
trees_rows_idx, trees_cols_idx = rasterio.transform.rowcol(
    transform,
    trees.X,
    trees.Y
)

In [ ]:
trees_df = pd.DataFrame({
    "row": trees_rows_idx,
    "col": trees_cols_idx,
    "z": trees.Z.values
})

Calculate Height Above Ground

In [ ]:
merged = trees_df.merge(
    mean_ground_grid,
    on=["row", "col"],
    how="left"
)

merged["HAG"] = merged["z"] - merged["ground_mean"]

Filter to include only unclassified points between 1 and 10 feet

In [ ]:
height_mask = ((merged.z > 1) & (merged.z < 10))
height_filtered = merged[height_mask]

Density Raster of unclassified points between 1 and 10 feet

In [ ]:
point_counts = height_filtered.groupby(["row", "col"]).size().reset_index(name="count")

Take vertical slices

In [ ]:
slice_height = 0.5

filtered["slice"] = (filtered["HAG"] // slice_height).astype(int)

In [ ]:
voxels = lower.groupby(["row", "col", "slice"]).size().reset_index(name="count")

vertical_persistence = voxels.groupby(["row", "col"]).size().reset_index(name="num_slices")

In [ ]:
trunk_candidates = vertical_persistence[vertical_persistence.num_slices >= 6]

ALL LIDAR Points

In [ ]:
rows, cols = rasterio.transform.rowcol(
    transform,
    cloud.X,
    cloud.Y
)